# Clustering Method Comparison — Mean-Pooled Embeddings

Runs all four clustering methods from `cluster_alts.py` on the pre-computed
`mean_pooled_embeddings.pkl` file and visualizes each result with t-SNE.

Clustering Algorithms Used:
1. Fuzzy K-Means
2. Spectral Clustering
3. DBSCAN
4. GMM

### Step 1: Setup (drive mounting, paths)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE           = "/content/drive/MyDrive/playlist-recommender"
EMBEDDINGS_PKL       = f"{DRIVE_BASE}/mean_pooled_embeddings.pkl"
CLUSTERS_OUTPUT_DIR  = f"{DRIVE_BASE}/clusters_alt"
REPO_DIR             = "/content/RecsysUpgrade"

import os
os.makedirs(CLUSTERS_OUTPUT_DIR, exist_ok=True)
print("Paths configured.")

### Step 2: Clone repo, install requirements.txt and import `cluster_alts`

In [ ]:
import os

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/siddmohanty111/RecsysUpgrade.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

print("Repo ready:", REPO_DIR)

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location(
    "cluster_alts",
    os.path.join(REPO_DIR, "PlaylistRecsysUpgrade", "clustering", "cluster_alts.py"),
)
cluster_alts = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cluster_alts)

print("cluster_alts loaded.")

In [ ]:
%pip install -r requirements.txt

#### cuML needs a special install

In [ ]:
pip install \
    --extra-index-url=https://pypi.nvidia.com \
    "cudf-cu12==26.4.*" "dask-cudf-cu12==26.4.*" "cuml-cu12==26.4.*" \
    "cugraph-cu12==26.4.*" "nx-cugraph-cu12==26.4.*" "cuxfilter-cu12==26.4.*" \
    "cucim-cu12==26.4.*" "pylibraft-cu12==26.4.*" "raft-dask-cu12==26.4.*" \
    "cuvs-cu12==26.4.*" "nx-cugraph-cu12==26.4.*"

### Step 3 — Load embeddings

In [ ]:
import pickle
import numpy as np

with open(EMBEDDINGS_PKL, 'rb') as f:
    data = pickle.load(f)

# Support both a raw dict (pid -> vector) and the structured dict used by other pipeline scripts
if isinstance(data, dict) and 'playlist_embeddings' in data:
    playlist_embeddings = data['playlist_embeddings']
    playlist_titles     = data.get('playlist_titles', {})
    playlist_tracks     = data.get('playlist_tracks', {})
else:
    # Assume the file is directly a pid -> embedding dict
    playlist_embeddings = data
    playlist_titles     = {}
    playlist_tracks     = {}

embedding_matrix = np.array(list(playlist_embeddings.values()))
pids = list(playlist_embeddings.keys())

print(f"Loaded {len(pids)} playlists, embedding dim = {embedding_matrix.shape[1]}")

### Step 4: Configuration

In [ ]:
NUM_CLUSTERS = 50  # single int — adjust as needed (50–200)

# Subsample for clustering (DBSCAN and Spectral are memory-intensive at scale)
CLUSTER_SAMPLE = min(50_000, len(pids))

# Subsample for t-SNE (always fast at ≤5k)
TSNE_SAMPLE = min(5_000, CLUSTER_SAMPLE)

rng = np.random.default_rng(42)
cluster_idx = rng.choice(len(pids), size=CLUSTER_SAMPLE, replace=False)
cluster_pids = [pids[i] for i in cluster_idx]
cluster_embeddings = {pid: playlist_embeddings[pid] for pid in cluster_pids}

print(f"Clustering {CLUSTER_SAMPLE} playlists into {NUM_CLUSTERS} clusters, t-SNE on {TSNE_SAMPLE}.")

### Step 5: DBSCAN

In [ ]:
# Using defaults for now

DBSCAN_EPS         = 0.5
DBSCAN_MIN_SAMPLES = 5

dbscan_output = os.path.join(CLUSTERS_OUTPUT_DIR, "dbscan_clusters.csv")

dbscan_labels = cluster_alts.dbscan(
    playlist_embeddings=cluster_embeddings,
    playlist_titles=playlist_titles,
    playlist_tracks=playlist_tracks,
    output_file=dbscan_output,
    eps=DBSCAN_EPS,
    min_samples=DBSCAN_MIN_SAMPLES,
)

n_found = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = (dbscan_labels == -1).sum()
print(f"DBSCAN done: {n_found} clusters, {n_noise} noise points. Results saved to {dbscan_output}")

### Step 6: Gaussian Mixture Model

In [ ]:
gaussianmix_output = os.path.join(CLUSTERS_OUTPUT_DIR, "gaussianmix_clusters.csv")

gaussianmix_labels = cluster_alts.gaussianmix(
    playlist_embeddings=cluster_embeddings,
    num_clusters=NUM_CLUSTERS,
    playlist_titles=playlist_titles,
    playlist_tracks=playlist_tracks,
    output_file=gaussianmix_output,
)

print(f"Gaussian Mixture done. Results saved to {gaussianmix_output}")

### Step 7: Fuzzy K-Means

In [ ]:
fkmeans_output = os.path.join(CLUSTERS_OUTPUT_DIR, "fkmeans_clusters.csv")

fkmeans_labels = cluster_alts.fkmeans(
    playlist_embeddings=cluster_embeddings,
    num_clusters=NUM_CLUSTERS,
    playlist_titles=playlist_titles,
    playlist_tracks=playlist_tracks,
    output_file=fkmeans_output,
)

print(f"Fuzzy K-Means done. Results saved to {fkmeans_output}")

### Step 8: t-SNE

In [ ]:
from sklearn.manifold import TSNE
import numpy as np

# cluster_embeddings is already a subsample; take TSNE_SAMPLE from it for t-SNE
cluster_matrix = np.array(list(cluster_embeddings.values()), dtype=np.float32)
rng_tsne = np.random.default_rng(0)
tsne_local_idx = rng_tsne.choice(len(cluster_matrix), size=TSNE_SAMPLE, replace=False)
sample_matrix = cluster_matrix[tsne_local_idx]

print(f"Running t-SNE on {TSNE_SAMPLE} samples...")
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_jobs=-1)
tsne_coords = tsne.fit_transform(sample_matrix)
print("t-SNE complete.")

### Step 9: Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm

methods = {
    "Fuzzy K-Means":    fkmeans_labels[tsne_local_idx],
    "DBSCAN":           dbscan_labels[tsne_local_idx],
    "Gaussian Mixture": gaussianmix_labels[tsne_local_idx],
}

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, (title, labels) in zip(axes, methods.items()):
    unique_labels = np.unique(labels)
    cmap = cm.get_cmap('tab20', max(len(unique_labels), 1))
    colors = np.array([cmap(np.where(unique_labels == l)[0][0]) if l != -1 else (0.7, 0.7, 0.7, 1.0)
                       for l in labels])
    ax.scatter(tsne_coords[:, 0], tsne_coords[:, 1],
               c=colors, s=4, alpha=0.6, linewidths=0)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle(f"Clustering Comparison — {NUM_CLUSTERS} clusters, n={CLUSTER_SAMPLE}",
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CLUSTERS_OUTPUT_DIR, "clustering_comparison_tsne.png"), dpi=150)
plt.show()
print("Plot saved.")

### Spectral 
Slow due to affinity matrix

In [ ]:
spectral_output = os.path.join(CLUSTERS_OUTPUT_DIR, "spectral_clusters.csv")

spectral_labels = cluster_alts.spectral(
    playlist_embeddings=cluster_embeddings,
    num_clusters=NUM_CLUSTERS,
    playlist_titles=playlist_titles,
    playlist_tracks=playlist_tracks,
    output_file=spectral_output,
)

print(f"Spectral Clustering done. Results saved to {spectral_output}")

# Visualize spectral result on the existing t-SNE projection
import matplotlib.pyplot as plt
import matplotlib.cm as cm

fig, ax = plt.subplots(figsize=(8, 6))
unique_labels = np.unique(spectral_labels)
cmap = cm.get_cmap('tab20', len(unique_labels))
colors = np.array([cmap(np.where(unique_labels == l)[0][0]) for l in spectral_labels[tsne_local_idx]])
ax.scatter(tsne_coords[:, 0], tsne_coords[:, 1], c=colors, s=4, alpha=0.6, linewidths=0)
ax.set_title("Spectral Clustering", fontsize=13, fontweight='bold')
ax.set_xlabel("t-SNE 1"); ax.set_ylabel("t-SNE 2")
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig(os.path.join(CLUSTERS_OUTPUT_DIR, "spectral_tsne.png"), dpi=150)
plt.show()
print("Spectral plot saved.")